# DDP Verify Call

Loads `.env.ddp` and checks IDVerse review status using `DDP_REQUEST_ID`.

In [3]:
# ============================================================
# CONFIGURATION - run this cell first
# ============================================================
import json
import os
from pathlib import Path

import requests
from dotenv import load_dotenv

ENV_FILE = ".env.ddp"

if not Path(ENV_FILE).exists():
    print(f"WARNING: {ENV_FILE} not found - create it with DDP_API_URL and API_KEY")
else:
    load_dotenv(ENV_FILE, override=True)
    print(f"Loaded {ENV_FILE}")

DDP_QUERY_API_URL = os.getenv("DDP_QUERY_API_URL") or "http://h-api.online-metrix.net/api/query"
DDP_ORG_ID = os.getenv("DDP_ORG_ID", "")
API_KEY = os.getenv("API_KEY", "")
DDP_REQUEST_ID = os.getenv("DDP_REQUEST_ID", "").strip()
if not DDP_REQUEST_ID:
    raise ValueError("DDP_REQUEST_ID is not set in .env.ddp")

def is_set(value):
    return "set" if value else "NOT SET"

print(f"DDP_QUERY_API_URL: {DDP_QUERY_API_URL or 'NOT SET'}")
print(f"DDP_ORG_ID      : {DDP_ORG_ID or 'NOT SET'}")
print(f"API_KEY     : {is_set(API_KEY)}")
print(f"Request ID  : {DDP_REQUEST_ID}")

Loaded .env.ddp
DDP_QUERY_API_URL: http://h-api.online-metrix.net/api/query
DDP_ORG_ID      : bmdcdpug
API_KEY     : set
Request ID  : fffd72f5-d3e8-473a-b54e-5cdea0b7c8e0


---
## Verify Call

In [4]:
# GET DDP_QUERY_API_URL - check review status using an existing DDP_REQUEST_ID
if not DDP_QUERY_API_URL:
    raise ValueError("DDP_QUERY_API_URL is not set")
if not DDP_ORG_ID:
    raise ValueError("DDP_ORG_ID is not set in .env.ddp")
if not API_KEY:
    raise ValueError("API_KEY is not set in .env.ddp")
if not DDP_REQUEST_ID:
    raise ValueError("DDP_REQUEST_ID is not set in .env.ddp")

params = {
    "org_id": DDP_ORG_ID,
    "api_key": API_KEY,
    "action": "check_review_status",
    "event_type": "init_auth",
    "request_id": DDP_REQUEST_ID,
}

display_params = dict(params)
display_params["api_key"] = "***" if API_KEY else ""
prepared = requests.Request("GET", DDP_QUERY_API_URL, params=display_params).prepare()
print("Request URL:")
print(prepared.url)

resp = requests.get(DDP_QUERY_API_URL, params=params, headers={"accept": "application/json"}, timeout=30)

print(f"\nStatus: {resp.status_code}")
try:
    data = resp.json()
    print(json.dumps(data, indent=2))
except ValueError:
    print(resp.text[:1000])

resp.raise_for_status()

Request URL:
http://h-api.online-metrix.net/api/query?org_id=bmdcdpug&api_key=%2A%2A%2A&action=check_review_status&event_type=init_auth&request_id=fffd72f5-d3e8-473a-b54e-5cdea0b7c8e0

Status: 200
input_request_id=fffd72f5%2dd3e8%2d473a%2db54e%2d5cdea0b7c8e0&policy_engine_version=17%2e1%2e3&policy_site_id=P1%2fotin119%2eprod%2esac&request_duration=11&request_id=bbe89140%2d73f8%2d4408%2da621%2d9c91fff2662d&request_id_activities=_CHALLENGED&request_id_activities=_CHALLENGE_FAILED&request_result=success&review_status=challenge
